## Create a custom AML environment (container) from Dockerfile

This notebook builds an Azure ML custom environment from `container/Dockerfile`
and `container/requirements.txt` using the AML Python SDK. The resulting
environment can be referenced by a compute-cluster training job (e.g.
`aml_cc_finetune_llama3.ipynb`) as `environment="<env_name>@latest"`.

### Workflow

The AML SDK's `BuildContext` works in two steps:

1. **Upload** the local build context folder (`./container`) to the
   workspace's default blob store — over HTTPS, authenticated as you with
   Entra ID (no storage keys).
2. **Trigger an ACR build** on the workspace's container registry, which
   pulls the uploaded folder, runs `docker build`, and publishes the image
   as a new environment version.

The challenge: this workspace's default storage account is **behind a managed
private endpoint**, so plain public access is blocked. We solve that with a
**Network Security Perimeter (NSP)** rule allowing this client's IPv4 in,
plus a subscription-scoped inbound rule for AML / ACR managed services.

### One-time setup checklist (per machine / per IP change)

| # | Step | Notes |
|---|------|-------|
| 1 | `az login --tenant <tenant>` | refresh if you see `AzureCliCredential` errors |
| 2 | **Turn off Global Secure Access (GSA) tunnel** | NSP is IPv4-only; GSA tunnels via Microsoft IPv6 → `AuthorizationFailure`. Verify: `netstat -rn \| grep ^default` should NOT show a `utun*` interface |
| 3 | Run **Step 1** — NSP + profile + inbound `/24` for current IP | idempotent; auto-removes stale `client-*` rules |
| 4 | Run **Step 2** — associate workspace storage with NSP (Enforced) | idempotent; first time is an async LRO |
| 5 | **Policy exemption** for `StorageAccount_DisableLocalAuth_Modify` | tenant-admin task; see the md cell labelled *Policy exemption*. Without it, Step 2b will revert silently and jobs will fail with `ForbiddenError` |
| 6 | Run **Step 2b** — adds `allow-aml-subscription` NSP rule + sets `allowSharedKeyAccess=true` + `publicNetworkAccess=Enabled` | idempotent; warns if shared-key reverts (= missing exemption) |
| 7 | Run **smoke-test** cell — wait until HTTP 200/400 instead of timeout | NSP propagation: ~60–120 s the first time |
| 8 | Run **env create** + **smoke job** — registers env, then triggers ACR build | image build is async, track in Studio or via the build-poll cell |

### Each subsequent run (same machine, same IP, no GSA)

Steps 3, 4, 6 are no-ops; just rerun **env create** + **smoke job**.

### Pre-flight commands

If something looks wrong, run these in a terminal:

```bash
# Is GSA tunnel routing traffic?
netstat -rn | grep "^default" | head -5
#  → no utun* line means GSA is off ✅

# Does authenticated storage work?
az storage container show \
  -n "azureml-blobstore-<workspace-guid>" \
  --account-name <storage-account-name> \
  --auth-mode login --query name
#  → prints the container name ✅
```

### Troubleshooting

If you see this in the very first cell that builds `MLClient`:
```
AzureCliCredential: Please run 'az login' to set up an account and login to the tenant
```
relogin from a shell:
```bash
az logout
az account clear
az login --tenant 00000000-0000-0000-0000-000000000000
```

If a submitted job fails with `Key based authentication is not permitted` →
the policy exemption is missing or got reverted; re-check Step 2b output and
Azure Portal → Policy → Exemptions.

Reference:
* Environment (BuildContext, Dockerfile): https://learn.microsoft.com/en-us/azure/machine-learning/how-to-manage-environments-v2?view=azureml-api-2&tabs=python
* Curated ACPT base images: https://onnxruntime.ai/docs/ecosystem/ptca_image_list.html

In [1]:
import azure.ai.ml as aml
print(f"Azure ML SDK version: {aml.__version__}")

Azure ML SDK version: 1.33.0


In [2]:
# get a handle to the workspace
import os
from azure.ai.ml import MLClient
from dotenv import load_dotenv

config_file_name = "germanywest.env"
config_file_path = os.path.join(".", "config", config_file_name)
load_dotenv(dotenv_path=config_file_path, override=True)

from utils.amlauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

ml_client = MLClient(
    credential, settings.subscription_id, settings.resource_group, settings.workspace
)

In [3]:
# get the location of the current workspace
workspace_location = ml_client.workspaces.get(settings.workspace).location
print(f"Workspace location: {workspace_location}")

Workspace location: germanywestcentral


### Wrap the storage account in a Network Security Perimeter (NSP)

The workspace's default storage account sits behind a managed private endpoint,
so its public `networkAcls` won't let traffic from this laptop reach the blob
data plane. The supported pattern (per
[NSP for Azure Storage](https://learn.microsoft.com/azure/storage/files/files-network-security-perimeter))
is to put the storage account into an **NSP** with an inbound access rule for
the client IP. Once associated in **Enforced** mode, NSP becomes the
authoritative inbound gate — the storage account's own firewall rules are no
longer evaluated, so we don't need to touch `networkAcls`.

NSPs are **regional**, so we create one in the same region as the workspace
storage (e.g. `germanywestcentral`).

The next three cells (idempotent — safe to re-run):

1. **Create / update NSP + profile + inbound IP rule**
   - NSP: `aml-build-clients-{region}`
   - Profile: `clients`
   - Rule: `client-{my_ip_dashed}` allowing the current public IPv4.
   - Stale `client-*` rules from previous IPs are removed.
2. **Associate the workspace storage account** with that profile in `Enforced`
   mode (async LRO — waits for completion).
3. **Smoke test** — poll the blob endpoint until NSP enforcement propagates
   (~60–120 s).

> **Note on Global Secure Access (GSA):** when GSA is enabled, your Azure
> egress is tunneled over IPv6 from a Microsoft-owned address. NSP currently
> only supports IPv4 rules, so storage will return `AuthorizationFailure`.
> Turn GSA off for this workflow.

In [4]:
# Step 1: create (or update) an NSP + profile + inbound access rule covering
# THIS client's current public IPv4 address (or a configurable CIDR range
# around it, so an ISP-issued IP change inside the same block doesn't require
# re-running this cell). Cleans up stale client-* rules from previous runs.
#
# Resources (in the same region as the workspace storage):
#   - NSP      : aml-build-clients-{region}
#   - Profile  : clients
#   - Rule     : client-{network_dashed}-{prefix}  e.g. client-83-171-168-0-24
#
# Tuneable:
#   CLIENT_CIDR_PREFIX = 32  → strict /32 (single IP)
#                        24  → cover the /24 around the current IP (recommended
#                              when your ISP rotates the last octet)
#                        16  → /16 (much larger; only if you really need it)
#
# Idempotent. Uses `credential` + `settings` from the AuthHelper cell.
#
# Caveat — Global Secure Access (GSA):
#   When GSA is enabled, traffic to Azure may egress over IPv6 from a
#   Microsoft-owned address. NSP currently supports IPv4 access rules only,
#   so storage will reject GSA-tunneled requests as `AuthorizationFailure`.
#   Turn GSA off before running the upload.

import ipaddress
import re
import requests
from azure.mgmt.storage import StorageManagementClient
from azure.mgmt.network import NetworkManagementClient
from azure.mgmt.network.models import (
    NetworkSecurityPerimeter,
    NspProfile,
    NspAccessRule,
)

# How tight should the inbound rule be? /32 = just my IP, /24 = my local block.
CLIENT_CIDR_PREFIX = 24

IPV4_RE = re.compile(r"^\d{1,3}(?:\.\d{1,3}){3}$")

def _get_client_ipv4() -> str:
    """Return this client's apparent public IPv4. Tries a couple of services."""
    for url in ("https://api.ipify.org", "https://ipv4.icanhazip.com"):
        try:
            ip = requests.get(url, timeout=5).text.strip()
            if IPV4_RE.match(ip):
                return ip
        except Exception:
            pass
    raise RuntimeError(
        "Could not determine an IPv4 public address. "
        "Disable Global Secure Access if it is forcing IPv6 egress."
    )

# Resolve storage account (need region — NSPs are regional)
ws = ml_client.workspaces.get(settings.workspace)
sa_name = ws.storage_account.rsplit("/", 1)[-1]
storage_mgmt = StorageManagementClient(credential, settings.subscription_id)
sa_props = storage_mgmt.storage_accounts.get_properties(settings.resource_group, sa_name)
sa_region = sa_props.location
sa_arm_id = sa_props.id
print(f"Storage account : {sa_name}  (region={sa_region})")

# Current public IPv4 → CIDR
my_ip = _get_client_ipv4()
my_cidr = ipaddress.ip_network(f"{my_ip}/{CLIENT_CIDR_PREFIX}", strict=False)
network_addr = str(my_cidr.network_address)
print(f"Client IPv4     : {my_ip}")
print(f"Rule CIDR       : {my_cidr}  ({my_cidr.num_addresses} addresses)")

# Network management client
net_client = NetworkManagementClient(credential, settings.subscription_id)

# Stable names (include prefix so /24 vs /32 vs new block don't collide)
nsp_name     = f"aml-build-clients-{sa_region}"
profile_name = "clients"
rule_name    = f"client-{network_addr.replace('.', '-')}-{CLIENT_CIDR_PREFIX}"

# 1a. NSP (create or update)
nsp = net_client.network_security_perimeters.create_or_update(
    settings.resource_group, nsp_name,
    NetworkSecurityPerimeter(location=sa_region, tags={"app": "aml-build", "owner": "notebook"}),
)
print(f"✅ NSP        : {nsp.name}")

# 1b. Profile (create or update)
profile = net_client.network_security_perimeter_profiles.create_or_update(
    settings.resource_group, nsp_name, profile_name,
    NspProfile(),
)
print(f"✅ Profile    : {profile.name}")

# 1c. Clean up stale client-* rules that don't match the current rule name
existing_rules = list(
    net_client.network_security_perimeter_access_rules.list(
        settings.resource_group, nsp_name, profile_name
    )
)
for r in existing_rules:
    if r.name and r.name.startswith("client-") and r.name != rule_name:
        net_client.network_security_perimeter_access_rules.delete(
            settings.resource_group, nsp_name, profile_name, r.name
        )
        print(f"🗑️  Removed stale rule: {r.name}  prefixes={r.address_prefixes}")

# 1d. Inbound access rule for current CIDR (create or update)
rule = net_client.network_security_perimeter_access_rules.create_or_update(
    settings.resource_group, nsp_name, profile_name, rule_name,
    NspAccessRule(direction="Inbound", address_prefixes=[str(my_cidr)]),
)
print(f"✅ Access rule: {rule.name}  direction={rule.direction}  prefixes={rule.address_prefixes}")

Storage account : amlwwywdos1036066399  (region=germanywestcentral)
Client IPv4     : 83.171.168.112
Rule CIDR       : 83.171.168.0/24  (256 addresses)
✅ NSP        : aml-build-clients-germanywestcentral
✅ Profile    : clients
✅ Access rule: client-83-171-168-0-24  direction=Inbound  prefixes=['83.171.168.0/24']


In [31]:
# Step 2: associate the workspace storage account with the NSP profile.
#
# This is an async (LRO) operation. Once it returns "Succeeded", NSP enforces
# the inbound IP rule on the storage account's data plane (blob, file, etc.).
#
# Idempotent: re-running the same association is accepted by the service.

from azure.mgmt.network.models import NspAssociation, SubResource

assoc_name = f"{sa_name}-to-{profile_name}"

print(f"Associating storage {sa_name} → NSP {nsp_name}/{profile_name} ...")
assoc_poller = net_client.network_security_perimeter_associations.begin_create_or_update(
    settings.resource_group, nsp_name, assoc_name,
    NspAssociation(
        private_link_resource=SubResource(id=sa_arm_id),
        profile=SubResource(id=profile.id),
        access_mode="Enforced",   # use "Learning" first if you want to audit before enforcing
    ),
)
assoc = assoc_poller.result()
print(f"✅ Association : {assoc.name}")
print(f"   accessMode  : {assoc.access_mode}")
print(f"   state       : {assoc.provisioning_state}")
print("\nWait ~60–120s for NSP enforcement to propagate, then re-run the smoke-test cell.")

Associating storage amlwwywdos1036066399 → NSP aml-build-clients-germanywestcentral/clients ...
✅ Association : amlwwywdos1036066399-to-clients
   accessMode  : Enforced
   state       : Succeeded

Wait ~60–120s for NSP enforcement to propagate, then re-run the smoke-test cell.


### Policy exemption — required so shared-key stays enabled

In Microsoft-internal tenants the management-group policy assignment
`MCAPSGovDeployPolicies` (Tenant Root) includes the
`StorageAccount_DisableLocalAuth_Modify` policy with a `modify` effect that
auto-flips `allowSharedKeyAccess` back to `false` on every storage account in
the tenant. Without an exemption, the cell below will appear to succeed but the
value will silently revert, and AML compute-cluster jobs will fail with:

```
Key based authentication is not permitted on this storage account
ErrorCode: ForbiddenError
```

**Exemption created for this workspace storage** (one-time, done from the
Azure Portal as tenant admin since RG-level Owner is not enough):

| Field | Value |
|---|---|
| Policy assignment | `MCAPSGovDeployPolicies` (scope: Tenant Root MG `787eb5ff-…`) |
| Policy reference | `StorageAccount_DisableLocalAuth_Modify` |
| Exemption scope | `/subscriptions/{sub}/resourceGroups/{rg}/providers/Microsoft.Storage/storageAccounts/{sa_name}` |
| Category | `Mitigated` |
| Justification | AML v2 Execution service performs `listKeys` on the workspace storage account during job orchestration on user-managed compute clusters. Mitigated by NSP IPv4 allow-list (cells above) + Entra RBAC for all OAuth data-plane access. |

CLI form (run as someone with `policyAssignments/exempt/action` on the MG):

```bash
az policy exemption create \
  --name "Allow-SharedKey-<sa-name>" \
  --display-name "AML compute-cluster needs listKeys on workspace storage" \
  --policy-assignment "/providers/Microsoft.Management/managementGroups/<tenant-mg>/providers/Microsoft.Authorization/policyAssignments/MCAPSGovDeployPolicies" \
  --policy-definition-reference-ids "StorageAccount_DisableLocalAuth_Modify" \
  --exemption-category "Mitigated" \
  --scope "/subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.Storage/storageAccounts/<sa-name>"
```

Once the exemption exists, the next cell (`Step 2b`) can flip
`allowSharedKeyAccess=true` and have it stick.

In [ ]:
# Step 2b: open the workspace storage for AML control plane.
#
# Once NSP is Enforced (cell above), three more things must be true so that the
# AML execution service, ACR Tasks, and your laptop can all do their jobs:
#
#   1. NSP profile must have an Inbound rule allowing the AML subscription
#      (covers AML/ACR managed services running inside your sub) — in addition
#      to the client/{ip} rule.
#   2. Storage `allowSharedKeyAccess=true` — AML v2 Execution service does
#      `listKeys` regardless of identity settings on the job. Without this,
#      jobs fail with "Key based authentication is not permitted on this
#      storage account" (ForbiddenError). Requires a Policy Exemption for
#      `StorageAccount_DisableLocalAuth_Modify` if MCAPS Gov is enforced.
#   3. Storage `publicNetworkAccess=Enabled` — NSP itself is the firewall now,
#      but the storage public endpoint must accept traffic from it.
#
# All three are idempotent; rerunning is a no-op when nothing needs to change.

from azure.mgmt.network.models import NspAccessRule, SubscriptionId
from azure.mgmt.storage.models import StorageAccountUpdateParameters

# 2b.1 — NSP inbound rule allowing AML services in this subscription
aml_rule_name = "allow-aml-subscription"
aml_rule = net_client.network_security_perimeter_access_rules.create_or_update(
    settings.resource_group, nsp_name, profile_name, aml_rule_name,
    NspAccessRule(
        direction="Inbound",
        subscriptions=[SubscriptionId(id=f"/subscriptions/{settings.subscription_id}")],
    ),
)
print(f"✅ NSP rule  : {aml_rule.name}  direction={aml_rule.direction}  subs={[s.id for s in (aml_rule.subscriptions or [])]}")

# 2b.2 — Storage shared-key + public network access
sa_state = storage_mgmt.storage_accounts.get_properties(settings.resource_group, sa_name)
print(f"   storage now: shared_key={sa_state.allow_shared_key_access}  public_network={sa_state.public_network_access}")

patch = {}
if sa_state.allow_shared_key_access is not True:
    patch["allow_shared_key_access"] = True
if str(sa_state.public_network_access) != "Enabled":
    patch["public_network_access"] = "Enabled"

if patch:
    print(f"Applying storage PATCH : {patch}")
    storage_mgmt.storage_accounts.update(
        settings.resource_group, sa_name,
        StorageAccountUpdateParameters(**patch),
    )
    after = storage_mgmt.storage_accounts.get_properties(settings.resource_group, sa_name)
    print(f"✅ storage now: shared_key={after.allow_shared_key_access}  public_network={after.public_network_access}")
    if after.allow_shared_key_access is not True:
        print("\n⚠️  shared-key still False — MCAPS Gov policy "
              "`StorageAccount_DisableLocalAuth_Modify` is reverting it. "
              "Create a Policy Exemption (Mitigated) for this storage account "
              "(see markdown cell above).")
else:
    print("✅ No storage change needed.")

In [5]:
# Smoke test: can we now reach the blob endpoint? Poll up to 90s for the
# NSP enforcement to propagate. We just need a TCP+TLS handshake — any HTTP
# response means we're through (e.g. 400 InvalidQueryParameterValue is fine,
# storage replied). A connect timeout means NSP hasn't propagated yet.
import time as _t
import urllib.error
import urllib.request
import ssl

blob_url = f"https://{sa_name}.blob.core.windows.net/"
ctx = ssl.create_default_context()
ok = False
for i in range(9):
    try:
        with urllib.request.urlopen(blob_url, timeout=5, context=ctx) as r:
            print(f"[{i*10:3d}s] HTTP {r.status} — reachable ✅")
            ok = True
            break
    except urllib.error.HTTPError as e:
        print(f"[{i*10:3d}s] HTTP {e.code} — reachable ✅")
        ok = True
        break
    except Exception as e:
        print(f"[{i*10:3d}s] not yet ({type(e).__name__}: {e}) — waiting ...")
        _t.sleep(10)

if not ok:
    print("\n❌ Still not reachable after 90s — NSP may need more time, "
          "or GSA is forcing IPv6 egress. Disable GSA and retry.")

[  0s] HTTP 400 — reachable ✅


### Build context

`BuildContext(path=...)` uploads the entire folder to the workspace's ACR and runs
`docker build` against the `Dockerfile` inside it. The folder must contain:

- `Dockerfile` — the build recipe (`container/Dockerfile`)
- `requirements.txt` — pip deps that the Dockerfile `COPY`s in (`container/requirements.txt`)

Any other files in the folder become part of the build context.

In [6]:
# Verify the build context folder exists and lists the expected files.
build_context_path = os.path.join(".", "container")
dockerfile_path   = os.path.join(build_context_path, "Dockerfile")
requirements_path = os.path.join(build_context_path, "requirements.txt")

assert os.path.isdir(build_context_path), f"Missing folder: {build_context_path}"
assert os.path.isfile(dockerfile_path),   f"Missing file: {dockerfile_path}"
assert os.path.isfile(requirements_path), f"Missing file: {requirements_path}"

print(f"Build context : {os.path.abspath(build_context_path)}")
for f in sorted(os.listdir(build_context_path)):
    full = os.path.join(build_context_path, f)
    size = os.path.getsize(full) if os.path.isfile(full) else 0
    print(f"  📄 {f}  ({size} bytes)")

Build context : /Users/yingding/Code/VCS/ai/model-fine-tuning/02-compute-cluster/container
  📄 Dockerfile  (962 bytes)
  📄 requirements.txt  (881 bytes)


### Create / update the custom environment

Each `create_or_update` call with a new `version` produces a new immutable version
of the environment. Reference it later from a `command` job as
`environment="<env_name>@latest"` or `environment="<env_name>:<version>"`.

In [ ]:
# Build a custom AML environment from container/Dockerfile + container/requirements.txt.
#
# WHY BuildContext (vs Environment(image=...)):
#   - image=...     : reuses an existing image as-is. Cannot add pip deps.
#   - BuildContext  : uploads the folder to the workspace ACR and runs `docker build`
#                     against the Dockerfile, producing a new image. This is the
#                     correct path when you need to layer custom pip packages
#                     (transformers, trl, peft, deepspeed, etc.) on top of an ACPT
#                     base image.
#
# The image build happens in the workspace's Azure Container Registry and may take
# several minutes the first time. Subsequent versions reuse cached layers when the
# Dockerfile / requirements.txt have not changed.
import datetime
from azure.ai.ml.entities import Environment, BuildContext

env_name        = "sft-finetune-cuda126"
env_version     = datetime.datetime.now().strftime("%Y%m%d.%H%M")
env_description = (
    "Custom ACPT-based fine-tuning env for Llama-3 / Phi: "
    "transformers + trl + peft + deepspeed + bitsandbytes."
)

custom_env = Environment(
    name=env_name,
    version=env_version,
    description=env_description,
    build=BuildContext(
        path=build_context_path,    # folder containing Dockerfile + requirements.txt
        dockerfile_path="Dockerfile",
    ),
    tags={"app": "finetuning", "domain": "ml", "base": "acpt-cu121-py310-torch230"},
)

print(f"Creating environment '{env_name}' version '{env_version}' ...")
start_time = datetime.datetime.now()
created_env = ml_client.environments.create_or_update(custom_env)
end_time = datetime.datetime.now()

print(f"✅ Submitted: {created_env.name}:{created_env.version}")
print(f"   id            : {created_env.id}")
print(f"   build context : {created_env.build.path if created_env.build else 'n/a'}")
print(f"   elapsed       : {end_time - start_time}")
print("\nNote: the actual image build runs asynchronously in ACR.")
print("Track progress in Azure ML Studio → Environments → "
      f"{env_name}:{env_version} → Build log.")

Creating environment 'sft-finetune-cuda126' version '20260602.1820' ...
✅ Submitted: sft-finetune-cuda126:20260602.1820
   id            : /subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourceGroups/rg-aml-yw-dos/providers/Microsoft.MachineLearningServices/workspaces/aml-ww-yw-dos/environments/sft-finetune-cuda126/versions/20260602.1820
   build context : https://amlwwywdos1036066399.blob.core.windows.net/azureml-blobstore-fb4fd209-d5ec-483a-9266-a270f4c2fd57/LocalUpload/693fd43482961b71fed886310044616cd8d4ccda71e1c73b12ae4e4499c9674f/container/
   elapsed       : 0:00:11.660405

Note: the actual image build runs asynchronously in ACR.
Track progress in Azure ML Studio → Environments → sft-finetune-cuda126:20260602.1820 → Build log.


### Trigger the ACR build via a smoke-test job

AML defers the actual image build until the environment is **first used by a
job or deployment**. Until then, there are zero ACR runs — that's why the
build-poll cell below would find nothing if run right after `create_or_update`.

The next cell submits a tiny `command` job that runs `python -c "..."` against
this environment on the AML compute cluster `Cluster-A100-1GPU`. This:

1. Causes AML to enqueue an **ACR build** of the env image (first time only;
   subsequent jobs reuse the cached image).
2. Once built, the job runs and prints **Python version, torch version,
   transformers version** so you can confirm what's actually in the container.

In [ ]:
# Submit a tiny smoke-test job that uses the newly-created env.
#
# Auth model on a user-managed GPU cluster:
#   - `identity=UserIdentityConfiguration()` makes the JOB RUNTIME use your
#     Entra token (the cluster MI impersonates you for data-plane reads).
#   - HOWEVER, the AML v2 Execution control plane still calls `listKeys` on
#     the workspace storage account during orchestration. So shared-key MUST
#     remain enabled on the storage (see Step 2b + the exemption md cell).
#   - Fully identity-only jobs are only possible on serverless compute, not on
#     a user-managed cluster.
#
# The job runs an inline `python -c "..."` on `Cluster-A100-1GPU` and prints
# Python / torch / transformers / trl / peft / bitsandbytes versions to verify
# what's actually inside the container once the build finishes.

from azure.ai.ml import command
from azure.ai.ml.entities import UserIdentityConfiguration

SMOKE_COMPUTE = "Cluster-A100-1GPU"
smoke_cmd = (
    "python -c \""
    "import sys, platform;\n"
    "print('Python   :', sys.version.split()[0]);\n"
    "print('Platform :', platform.platform());\n"
    "try:\n"
    "    import torch; print('torch    :', torch.__version__, 'cuda?', torch.cuda.is_available());\n"
    "except Exception as e: print('torch    : MISSING', e);\n"
    "try:\n"
    "    import transformers; print('transformers:', transformers.__version__);\n"
    "except Exception as e: print('transformers: MISSING', e);\n"
    "try:\n"
    "    import trl; print('trl      :', trl.__version__);\n"
    "except Exception as e: print('trl      : MISSING', e);\n"
    "try:\n"
    "    import peft; print('peft     :', peft.__version__);\n"
    "except Exception as e: print('peft     : MISSING', e);\n"
    "try:\n"
    "    import bitsandbytes; print('bitsandbytes:', bitsandbytes.__version__);\n"
    "except Exception as e: print('bitsandbytes: MISSING', e);\n"
    "\""
)

smoke_job = command(
    code=None,  # no source files needed for an inline -c command
    command=smoke_cmd,
    environment=f"{env_name}@latest",
    compute=SMOKE_COMPUTE,
    identity=UserIdentityConfiguration(),  # runtime identity = your Entra token
    display_name=f"env-smoke-{env_name}-{env_version}",
    experiment_name="env-smoke-tests",
    description=f"Triggers the first ACR build of {env_name}:{env_version} and prints versions.",
)

print(f"Submitting smoke job for env '{env_name}@latest' on '{SMOKE_COMPUTE}' ...")
print("Runtime auth: UserIdentityConfiguration  |  Control plane: shared-key (per exemption)")
submitted = ml_client.jobs.create_or_update(smoke_job)
print(f"✅ Job submitted")
print(f"   name        : {submitted.name}")
print(f"   status      : {submitted.status}")
print(f"   studio URL  : {submitted.studio_url}")
print("\nNext steps:")
print("  • Run the build-poll cell below to watch the ACR build.")
print("  • Once the build is Succeeded, the job will start running.")
print("  • Stream job logs in Studio (link above) or with:")
print(f"      ml_client.jobs.stream(\"{submitted.name}\")")

Submitting smoke job for env 'sft-finetune-cuda126@latest' on 'Cluster-A100-1GPU' ...
Auth: UserIdentityConfiguration (no shared-key)
✅ Job submitted
   name        : green_oyster_zsgrcwqwj3
   status      : Starting
   studio URL  : https://ml.azure.com/runs/green_oyster_zsgrcwqwj3?wsid=/subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourcegroups/rg-aml-yw-dos/workspaces/aml-ww-yw-dos&tid=787eb5ff-f2ee-4965-8074-edbba3402c84

Next steps:
  • Run the build-poll cell below to watch the ACR build.
  • Once the build is Succeeded, the job will start running.
  • Stream job logs in Studio (link above) or with:
      ml_client.jobs.stream("green_oyster_zsgrcwqwj3")


In [17]:
# Watch the ACR build: poll the environment's provisioning_state, locate the
# underlying ACR build run, and print the tail of its log.
#
# WHY a separate SDK package:
#   In `azure-mgmt-containerregistry` v15+, the `runs` / `tasks` operations
#   were moved into a separate preview package:
#     pip install --pre azure-mgmt-containerregistrytasks==1.0.0b1
#   Use `ContainerRegistryTasksMgmtClient.runs` to list, get, and fetch logs.
#
# Uses `created_env`, `ml_client`, `credential`, `settings` from earlier cells.

import time
import requests
from azure.mgmt.containerregistrytasks import ContainerRegistryTasksMgmtClient

# 1. Discover the workspace ACR.
ws = ml_client.workspaces.get(settings.workspace)
acr_arm_id = ws.container_registry
if not acr_arm_id:
    raise RuntimeError(
        "Workspace has no container_registry attached. Wait a minute after the "
        "first env build (AML auto-provisions one) and re-run."
    )
acr_name = acr_arm_id.rsplit("/", 1)[-1]
acr_rg   = acr_arm_id.split("/")[4]
print(f"Workspace ACR : {acr_name}  (rg={acr_rg})")

acr_tasks = ContainerRegistryTasksMgmtClient(
    credential=credential, subscription_id=settings.subscription_id
)

# 2. Poll env state + find matching ACR run.
print(f"\nPolling env '{env_name}:{env_version}' ...")
acr_run = None
for i in range(60):  # up to ~30 min
    env_now = ml_client.environments.get(env_name, env_version)
    state = getattr(env_now, "provisioning_state", None) or "?"
    image = getattr(env_now, "image", None) or "(pending)"
    print(f"  [{i*30:4d}s] env state={state}  image={image}")

    # Try to locate the matching ACR run if not yet found.
    if acr_run is None:
        for run in acr_tasks.runs.list(acr_rg, acr_name, top=20):
            imgs = run.output_images or []
            tag = (imgs[0].tag or "") if imgs else ""
            repo = (imgs[0].repository or "") if imgs else ""
            hay = f"{repo}:{tag}"
            if env_name in hay and env_version in hay:
                acr_run = acr_tasks.runs.get(acr_rg, acr_name, run.run_id)
                print(f"  → Matched ACR run: {acr_run.run_id}  status={acr_run.status}  image={hay}")
                break
    else:
        # Refresh run status
        acr_run = acr_tasks.runs.get(acr_rg, acr_name, acr_run.run_id)
        print(f"     ACR run status: {acr_run.status}")

    if str(state).lower() in ("succeeded", "failed", "canceled"):
        break
    if acr_run and str(acr_run.status).lower() in ("succeeded", "failed", "canceled", "timeout", "error"):
        break
    time.sleep(30)

# 3. Fetch the build log via the SAS URL.
if acr_run:
    log_result = acr_tasks.runs.get_log_sas_url(acr_rg, acr_name, acr_run.run_id)
    # `log_link` is the classic SAS URL; `log_artifact_link` is set when the
    # registry/task uses a log template. Prefer log_link, fall back to artifact.
    log_url = log_result.log_link or log_result.log_artifact_link
    if not log_url:
        print("⚠️  Run has neither log_link nor log_artifact_link.")
    else:
        kind = "log_link" if log_result.log_link else "log_artifact_link"
        print(f"\nACR log SAS URL ({kind}): {log_url[:100]}...")
        try:
            log_text = requests.get(log_url, timeout=30).text
            tail = log_text.splitlines()[-200:]
            print(f"\n──── ACR build log (last {len(tail)} lines) ────")
            print("\n".join(tail))
        except Exception as e:
            print(f"⚠️  Could not download log content: {e}")
else:
    print(f"\n⚠️  No matching ACR run found. Check Studio → Environments → "
          f"{env_name}:{env_version} → Build log.")

Workspace ACR : fb4fd209d5ec483a9266a270f4c2fd57  (rg=rg-aml-yw-dos)

Polling env 'sft-finetune-cuda126:20260602.1744' ...
  [   0s] env state=?  image=(pending)


KeyboardInterrupt: 

### Verify the environment registration

List all versions of the new environment and grab the latest one.

In [25]:
# List all versions of the new env and show the latest.
versions = list(ml_client.environments.list(name=env_name))
print(f"Found {len(versions)} version(s) for '{env_name}':")
for e in versions:
    print(f"  - {e.name}:{e.version}")

latest_env = ml_client.environments.get(name=env_name, label="latest")
print(f"\nLatest : {latest_env.name}:{latest_env.version}")
print(f"id     : {latest_env.id}")

Found 1 version(s) for 'sft-finetune-cuda126':
  - sft-finetune-cuda126:20260601.2113

Latest : sft-finetune-cuda126:20260601.2113
id     : /subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourceGroups/rg-aml-yw-dos/providers/Microsoft.MachineLearningServices/workspaces/aml-ww-yw-dos/environments/sft-finetune-cuda126/versions/20260601.2113


In [ ]:
# Optional: archive an old version (cannot be deleted, only archived).
# ml_client.environments.archive(name=env_name, version="<old_version>")

# Optional: restore an archived version.
# ml_client.environments.restore(name=env_name, version="<old_version>")